# 🌲 PINE 2기 7회차 — AI 에이전트 개발 2차

> **유스AI프로젝트:D** | 마포청소년문화의집 × KB데이타시스템 × Microsoft  
> 사용 기술: **Microsoft Agent Framework (MAF) 1.0.0** + Azure OpenAI (API Key)

---

## 🗓 오늘의 여정

```
[10:00~10:15] 🔁 오프닝 & 지난 회차 복습
[10:15~10:45] 🤖 Part 1: 멀티 에이전트 & GroupChat
              └─ 🔴 Quiz 1 (10분)
[10:55~11:25] 🧠 Part 2: 세션 기반 메모리 관리
              └─ 🔴 Quiz 2 (10분)
[11:35~12:15] 📚 Part 3: RAG 에이전트 개발
              └─ 🔴 Quiz 3 (15분)
[12:30~13:00] 🎤 팀 발표 & Q&A & 해커톤 예고
```

## 📌 지난 회차(6회차) 복습
- ✅ Azure OpenAI API 연결
- ✅ 단일 에이전트 + Tool 사용 (ReAct 패턴)
- ✅ 수학 문제 해결 에이전트

오늘은 **여러 에이전트가 협력**하고, **기억**하고, **외부 지식을 검색**하도록 만들어봅니다! 🚀

---
# ⚙️ 섹션 0: 환경 설정

**이 섹션을 가장 먼저 실행하세요!**  
Jupyter에서는 셀에서 `await`를 직접 사용할 수 있습니다.

In [ ]:
# 패키지 설치 확인 (Codespace 첫 실행 시 또는 수동 설치 필요 시)
# !pip install agent-framework==1.0.0 python-dotenv scikit-learn numpy -q

In [ ]:
# =============================================
# 📦 MAF 1.0.0 임포트
# =============================================
import os
import asyncio
from dotenv import load_dotenv

# MAF 핵심 임포트
from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatClient      # Azure OpenAI + API Key 지원
from agent_framework.orchestrations import GroupChatBuilder  # 멀티 에이전트

# .env 파일 로드
load_dotenv()

print("✅ 임포트 완료!")
print("📌 agent-framework 버전 확인:")
import importlib.metadata
print("  ", importlib.metadata.version("agent-framework"))

In [ ]:
# =============================================
# 🔑 Azure OpenAI 클라이언트 초기화 (API Key 방식)
# =============================================
# OpenAIChatClient는 환경변수를 자동으로 읽습니다:
#   AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, AZURE_OPENAI_MODEL
# 또는 직접 파라미터로 전달할 수도 있습니다.

chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_MODEL"],          # 배포 이름 (deployment name)
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview"),
)

print("✅ Azure OpenAI 클라이언트 초기화 완료!")
print(f"   모델: {os.environ['AZURE_OPENAI_MODEL']}")

In [ ]:
# =============================================
# 🧪 연결 테스트
# =============================================
# Jupyter에서는 await를 셀에서 직접 사용할 수 있습니다!

test_agent = Agent(
    client=chat_client,
    name="테스트봇",
    instructions="당신은 친절한 AI 어시스턴트입니다."
)

result = await test_agent.run("한 문장으로 자기소개 해줘!")
print("🤖 연결 성공!")
print("응답:", result)

---
# 🤖 Part 1: 멀티 에이전트 & GroupChat

## 1-1. 왜 멀티 에이전트가 필요할까?

```
단일 에이전트:     사용자 → [에이전트 1개] → 답변
                          (모든 역할을 혼자 담당)

멀티 에이전트:     사용자 → [GroupChat Manager]
                                  │
                    ┌─────────────┼─────────────┐
                    ▼             ▼             ▼
              [작가 에이전트] [편집자] [팩트체커]
                서로의 출력을 보며 협력 → 더 나은 결과!
```

## 1-2. MAF의 GroupChatBuilder

MAF 1.0.0에서는 `GroupChatBuilder`로 멀티 에이전트를 쉽게 구성합니다.

```python
from agent_framework.orchestrations import GroupChatBuilder

workflow = (
    GroupChatBuilder()
    .set_select_speakers_func(speaker_selector)  # 다음 발언자 결정 함수
    .participants([agent1, agent2, agent3])       # 참여 에이전트 목록
    .build()
)
```

## 1-3. 발언자 선택 함수 (Speaker Selector)

GroupChat의 핵심! 누가 다음에 발언할지 결정하는 함수입니다.

```python
def my_selector(state) -> str | None:
    # state 딕셔너리에는:
    #   state["participants"]  → 에이전트 이름 목록
    #   state["round_index"]   → 현재 라운드 번호 (0부터 시작)
    #   state["conversation"]  → 지금까지의 대화 이력
    
    if state["round_index"] >= 6:  # 6라운드 후 종료
        return None                # None 반환 = 종료
    
    names = list(state["participants"].keys())
    return names[state["round_index"] % len(names)]  # 순서대로 돌아가며
```

In [ ]:
# =============================================
# 🎮 예제: 선생님-학생 GroupChat
# =============================================

# 두 에이전트 정의
teacher = Agent(
    client=chat_client,
    name="선생님",
    instructions="""당신은 친절한 AI 선생님입니다.
개념을 쉽고 재미있게 설명하고, 학생의 이해를 돕는 질문을 던집니다.
답변은 3~4문장으로 간결하게 작성하세요."""
)

student = Agent(
    client=chat_client,
    name="학생",
    instructions="""당신은 AI를 배우는 호기심 많은 고등학생입니다.
모르는 것은 솔직하게 질문하고, 이해한 내용을 자기 말로 표현합니다.
답변은 2~3문장으로 짧게 작성하세요."""
)

# 라운드로빈 발언자 선택 함수 (선생님 → 학생 → 선생님 → 학생...)
MAX_ROUNDS = 4  # 총 발언 횟수

def round_robin_selector(state) -> str | None:
    if state["round_index"] >= MAX_ROUNDS:
        return None  # 종료
    names = list(state["participants"].keys())
    return names[state["round_index"] % len(names)]

# GroupChat 워크플로우 구성
teacher_student_chat = (
    GroupChatBuilder()
    .set_select_speakers_func(round_robin_selector)
    .participants([teacher, student])
    .build()
)

print("✅ GroupChat 워크플로우 구성 완료!")

In [ ]:
# =============================================
# ▶️ GroupChat 실행 헬퍼 함수
# =============================================

async def run_groupchat(workflow, task: str):
    """GroupChat 워크플로우를 실행하고 대화를 출력합니다."""
    print(f"📢 주제: {task}")
    print("=" * 60)

    all_messages = []
    async for event in workflow.run(task, stream=True):
        if event.type == "output":
            data = event.data
            # 스트리밍 중 개별 에이전트 응답
            if hasattr(data, "text") and hasattr(data, "author_name"):
                author = data.author_name or "?"
                print(f"\n🤖 [{author}]:")
                print(data.text)
                print("-" * 40)
                all_messages.append(data)
            # 최종 메시지 리스트
            elif isinstance(data, list):
                all_messages = data

    print(f"\n✅ 총 {len(all_messages)}개 메시지 완료")
    return all_messages


# 실행!
messages = await run_groupchat(
    teacher_student_chat,
    "멀티 에이전트가 무엇인지 설명해주세요!"
)

---
## 🔴 Quiz 1: 나만의 역할 에이전트 설계

### 📋 목표
**학생(Student)**, **전문가(Expert)**, **비평가(Critic)** 3개 에이전트를 설계하고,  
"RAG 에이전트란 무엇인가?" 주제로 GroupChat을 실행하세요.

### ⏱ 제한시간: 10분

### 성공 기준
- 3개 에이전트가 각각 다른 성격을 가진다
- GroupChat이 최소 6라운드 실행된다 (에이전트당 2번)
- 각 에이전트의 응답이 역할에 맞게 다르게 나온다

In [ ]:
# ============================================================
# 🔴 Quiz 1: 나만의 역할 에이전트 설계
# ============================================================
# 📋 목표: Student·Expert·Critic 3개 에이전트 GroupChat 구성
# ⏱  제한시간: 10분
# TODO: ??? 부분을 채워서 완성하세요!
# ============================================================

# 1️⃣ 학생 에이전트
quiz1_student = Agent(
    client=chat_client,
    name="학생",
    instructions="""???
    # 힌트: AI를 처음 배우는 고등학생, 질문이 많음, 짧게 2문장으로
    """
)

# 2️⃣ 전문가 에이전트
quiz1_expert = Agent(
    client=chat_client,
    name="전문가",
    instructions="""???
    # 힌트: AI 분야 경력자, 정확한 설명, 쉬운 비유 사용, 4~5문장
    """
)

# 3️⃣ 비평가 에이전트
quiz1_critic = Agent(
    client=chat_client,
    name="비평가",
    instructions="""???
    # 힌트: 논리적 허점 지적, 개선 방향 제시, 3문장
    """
)

# 4️⃣ 발언자 선택 함수 (3개 에이전트 라운드로빈)
QUIZ1_MAX = 6  # 총 6라운드 = 에이전트당 2번씩 발언

def quiz1_selector(state) -> str | None:
    if state["round_index"] >= ???:  # 언제 종료할까요?
        return None
    names = list(state["participants"].keys())
    return names[???]  # 어떻게 순서를 정할까요?

# 5️⃣ GroupChat 구성
quiz1_chat = (
    GroupChatBuilder()
    .set_select_speakers_func(???)
    .participants([???, ???, ???])
    .build()
)

# 6️⃣ 실행!
quiz1_messages = await run_groupchat(
    quiz1_chat,
    "RAG 에이전트란 무엇인가요? 오늘 배울 내용을 함께 정리해봐요!"
)

In [ ]:
# ✅ 정답 예시 (강사 참고용 — 학생은 먼저 직접 풀어보세요!)

# quiz1_student = Agent(
#     client=chat_client, name="학생",
#     instructions="""당신은 AI에 관심 있는 고등학교 2학년 학생입니다.
# 전문 용어가 나오면 솔직하게 "그게 뭐예요?"라고 질문합니다.
# 이해한 내용을 자기만의 비유로 표현하기도 합니다. 2문장으로 짧게."""
# )
# quiz1_expert = Agent(
#     client=chat_client, name="전문가",
#     instructions="""당신은 AI 분야 10년 경력 엔지니어입니다.
# 기술적으로 정확하되 고등학생도 이해할 수 있게 일상적 비유를 씁니다.
# 실제 사용 사례를 한 가지 이상 언급합니다. 4~5문장."""
# )
# quiz1_critic = Agent(
#     client=chat_client, name="비평가",
#     instructions="""당신은 비판적 사고를 하는 논리학자입니다.
# 앞선 설명의 논리적 허점이나 빠진 내용을 지적하고 개선 방향을 제시합니다. 3문장."""
# )
# def quiz1_selector(state):
#     if state["round_index"] >= 6: return None
#     return list(state["participants"].keys())[state["round_index"] % 3]
# quiz1_chat = GroupChatBuilder().set_select_speakers_func(quiz1_selector).participants([quiz1_student, quiz1_expert, quiz1_critic]).build()
# quiz1_messages = await run_groupchat(quiz1_chat, "RAG 에이전트란 무엇인가요?")

---
# 🧠 Part 2: 세션 기반 메모리 관리

## 2-1. LLM은 기본적으로 기억이 없다

```
[호출 1] 사용자: "내 이름은 지민이야"  → AI: "안녕하세요, 지민님!"
[호출 2] 사용자: "내 이름이 뭐야?"    → AI: "죄송하지만 모릅니다..." 😅
```

## 2-2. MAF의 Session으로 해결!

MAF 1.0.0은 **Session** 객체로 대화 이력을 자동으로 관리합니다.

```python
# 세션 생성 (대화 이력 저장소)
session = await agent.create_session()

# 같은 session 객체를 계속 전달 → AI가 이전 대화를 기억!
r1 = await agent.run("내 이름은 지민이야", session=session)
r2 = await agent.run("내 이름이 뭐야?",   session=session)  # 기억함! ✅
```

## 2-3. 세션 없이 vs 세션 있이 비교

In [ ]:
# =============================================
# 🧠 메모리 비교 실험: 세션 없이 vs 세션 있이
# =============================================

memory_agent = Agent(
    client=chat_client,
    name="기억테스트봇",
    instructions="당신은 학생의 정보를 기억하고 맞춤형 도움을 주는 AI 학습 도우미입니다."
)

print("=" * 50)
print("❌ 세션 없이 (기억 안 됨)")
print("=" * 50)

# 세션 없이: 매번 새로 시작
await memory_agent.run("안녕! 나는 고2 민준이야. 수학이 약해.")
r_no_session = await memory_agent.run("내가 누구라고 했지?")
print(f"🤖 (세션 없음): {r_no_session}")

print()
print("=" * 50)
print("✅ 세션 있이 (기억 됨!)")
print("=" * 50)

# 세션 생성
my_session = await memory_agent.create_session()

# 같은 session 전달 → 이전 대화를 기억!
await memory_agent.run("안녕! 나는 고2 민준이야. 수학이 약해.", session=my_session)
r_with_session = await memory_agent.run("내가 누구라고 했지?", session=my_session)
print(f"🤖 (세션 있음): {r_with_session}")

In [ ]:
# =============================================
# 🧪 다중 턴 세션 대화 예제
# =============================================

planner = Agent(
    client=chat_client,
    name="공부플래너",
    instructions="""당신은 고등학생 전문 공부 플래너 AI입니다.
학생의 이름, 약한 과목, 시험 날짜, 목표 점수를 파악하고 기억하세요.
매 답변에서 학생의 이름을 불러주고 맞춤형 계획을 제안합니다.
격려하는 말을 자주 해주세요."""
)

# 세션 생성
planner_session = await planner.create_session()

conversations = [
    "안녕! 나는 고2 서연이야. 수학 기말고사가 3주 후인데 현재 점수가 45점이야. 80점이 목표야!",
    "어떤 단원부터 시작해야 할까?",
    "내 이름이랑 목표가 뭐라고 했지?",  # 기억 확인!
]

print("📓 공부 플래너 세션 대화")
print("=" * 55)

for msg in conversations:
    print(f"\n👤 학생: {msg}")
    response = await planner.run(msg, session=planner_session)
    print(f"🤖 플래너: {response}")
    print("-" * 40)

---
## 🔴 Quiz 2: 기억하는 공부 플래너 에이전트

### 📋 목표
**이름, 약한 과목, 시험 날짜**를 기억하고 맞춤형 공부 계획을 제안하는  
나만의 플래너 에이전트를 Session을 사용해서 완성하세요.

### ⏱ 제한시간: 10분

### 성공 기준
- `create_session()`으로 세션 생성
- 3번 이상 대화
- 3번째 대화에서 첫 번째에 말한 이름/과목을 정확히 기억

In [ ]:
# ============================================================
# 🔴 Quiz 2: 기억하는 공부 플래너 에이전트
# ============================================================
# 📋 목표: 세션으로 이름·과목·목표를 기억하는 에이전트 완성
# ⏱  제한시간: 10분
# TODO: ??? 부분을 채워서 완성하세요!
# ============================================================

# 1️⃣ 나만의 공부 플래너 에이전트 정의
my_planner = Agent(
    client=chat_client,
    name="나의공부플래너",
    instructions="""???
    # 힌트: 학생 정보 기억, 이름 불러주기, 맞춤형 계획, 격려
    """
)

# 2️⃣ 세션 생성
my_planner_session = await my_planner.???()

print("🎓 나의 공부 플래너 에이전트 시작!")
print("=" * 55)

# 3️⃣ 대화 1: 자기소개 (이름 + 약한 과목 + 시험 날짜 포함!)
msg1 = "???"  # 예: "안녕! 나는 고2 ○○야. 영어가 약하고 다음달 3일 기말고사야."
r1 = await my_planner.run(msg1, ???=my_planner_session)
print(f"\n👤 나: {msg1}")
print(f"🤖 플래너: {r1}")

# 4️⃣ 대화 2: 구체적인 도움 요청
msg2 = "???"  # 예: "오늘부터 어떻게 공부하면 될까?"
r2 = await my_planner.run(msg2, ???=my_planner_session)
print(f"\n👤 나: {msg2}")
print(f"🤖 플래너: {r2}")

# 5️⃣ 대화 3: 기억 테스트! 처음에 말한 정보를 기억하는지?
msg3 = "내가 어떤 과목이 약하고, 시험이 언제라고 했지?"
r3 = await my_planner.run(msg3, ???=my_planner_session)
print(f"\n👤 나: {msg3}")
print(f"🤖 플래너: {r3}")

In [ ]:
# ✅ 정답 예시 (강사 참고용)

# my_planner = Agent(
#     client=chat_client, name="나의공부플래너",
#     instructions="""당신은 고등학생 전문 공부 플래너 AI입니다.
# 학생의 이름, 약한 과목, 시험 날짜, 목표 점수를 반드시 기억하세요.
# 매 답변에서 학생의 이름을 불러주고, 이전 정보를 활용해 맞춤 계획을 제안합니다.
# 항상 격려의 말로 마무리하세요."""
# )
# my_planner_session = await my_planner.create_session()
# msg1 = "안녕! 나는 고2 지수야. 영어가 약해서 현재 50점이야. 다음달 3일 기말고사에서 75점이 목표야!"
# r1 = await my_planner.run(msg1, session=my_planner_session)

---
# 📚 Part 3: RAG 에이전트 개발

## 3-1. RAG란?

**RAG (Retrieval-Augmented Generation)** = 검색 + 생성

```
일반 LLM의 한계:
  질문: "PINE 프로그램이 뭐야?"
  답변: "모릅니다" (학습 데이터에 없음)

RAG 방식:
  1️⃣ 질문: "PINE 프로그램이 뭐야?"
  2️⃣ 검색: rag_source.md에서 관련 청크 찾기
  3️⃣ @tool: 검색 결과를 에이전트가 자동으로 활용
  4️⃣ 답변: "PINE은 마포청소년문화의집과 Microsoft가..."
```

## 3-2. MAF의 @tool 데코레이터

MAF 1.0.0에서 RAG는 `@tool`로 감싼 검색 함수로 구현합니다!

```python
from agent_framework import tool

@tool(approval_mode="never_require")  # 자동 실행 (승인 불필요)
def search_knowledge_base(query: str) -> str:
    """지식베이스에서 관련 정보를 검색합니다."""
    # 검색 로직...
    return "검색 결과"

rag_agent = Agent(
    client=chat_client,
    instructions="...",
    tools=[search_knowledge_base]  # 에이전트에 도구 등록!
)
```

에이전트가 필요하다고 판단하면 **자동으로 `search_knowledge_base`를 호출**합니다!

In [ ]:
# =============================================
# 📚 Step 1: 지식베이스 로드 & 청크 분할
# =============================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def load_chunks(filepath: str, min_len: int = 80) -> list:
    """마크다운 파일을 '---' 기준으로 섹션 분할."""
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    sections = [s.strip() for s in text.split("---") if s.strip()]
    return [s for s in sections if len(s) > min_len]


# 지식베이스 로드
chunks = load_chunks("data/rag_source.md")

# TF-IDF 벡터화 (한국어용 문자 n-gram)
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=5000)
chunk_vectors = vectorizer.fit_transform(chunks)

print(f"✅ {len(chunks)}개 청크 로드 & 벡터화 완료!")
print(f"\n📄 첫 번째 청크 미리보기:")
print(chunks[0][:120], "...")

In [ ]:
# =============================================
# 📚 Step 2: @tool로 검색 함수 정의
# =============================================

@tool(approval_mode="never_require")
def search_knowledge_base(query: str) -> str:
    """진로, 학습, AI 직업, PINE 프로그램에 관한 지식베이스를 검색합니다.
    
    Args:
        query: 검색할 내용 (한국어)
    Returns:
        관련 정보 텍스트 (없으면 안내 메시지)
    """
    # 1️⃣ 쿼리 벡터화
    query_vec = vectorizer.transform([query])
    
    # 2️⃣ 코사인 유사도로 top-3 검색
    sims = cosine_similarity(query_vec, chunk_vectors)[0]
    top_idx = np.argsort(sims)[::-1][:3]
    
    # 3️⃣ 결과 조합
    results = [(chunks[i], sims[i]) for i in top_idx if sims[i] > 0]
    
    if not results:
        return "관련 정보를 찾지 못했습니다."
    
    parts = []
    for i, (chunk, score) in enumerate(results):
        parts.append(f"[참고 자료 {i+1}] (관련도: {score:.2f})\n{chunk[:400]}")
    
    return "\n\n".join(parts)


print("✅ @tool 함수 등록 완료!")

# 검색 함수 직접 테스트
test_result = search_knowledge_base("AI 관련 직업")
print("\n🔍 검색 테스트:")
print(test_result[:300], "...")

In [ ]:
# =============================================
# 📚 Step 3: RAG 에이전트 생성 & 실행
# =============================================

rag_agent = Agent(
    client=chat_client,
    name="진로상담사",
    instructions="""당신은 고등학생의 진로와 학습을 돕는 AI 상담사입니다.
질문에 답할 때 반드시 search_knowledge_base 도구를 사용하여 관련 정보를 검색하세요.
검색된 참고 자료를 바탕으로 답하고, 답변 마지막에 [출처: 참고 자료 N]을 표시하세요.
참고 자료에 없는 내용은 "확인이 필요합니다"라고 말하세요.""",
    tools=[search_knowledge_base]  # @tool 등록!
)

# 질문 1: AI 직업
q1 = "AI 관련 직업에는 어떤 것들이 있나요?"
print(f"\n{'='*60}")
print(f"❓ {q1}")
a1 = await rag_agent.run(q1)
print(f"🤖 진로상담사:\n{a1}")

# 질문 2: PINE 프로그램
q2 = "PINE 프로그램 수료 후 어떤 결과물을 얻을 수 있나요?"
print(f"\n{'='*60}")
print(f"❓ {q2}")
a2 = await rag_agent.run(q2)
print(f"🤖 진로상담사:\n{a2}")

---
## 🔴 Quiz 3: 진로 상담 RAG 에이전트 완성

### 📋 목표
나만의 RAG 에이전트를 설계하고, 진로 관련 질문 3개 이상에 답하세요.  
🌟 **보너스**: RAG + 세션 메모리를 결합해서 대화 맥락도 유지하세요!

### ⏱ 제한시간: 15분

### 성공 기준
- `@tool`로 search_knowledge_base를 에이전트에 등록
- 3개 이상의 다른 진로 질문에 답
- 🌟 보너스: Session으로 대화 기억 유지

In [ ]:
# ============================================================
# 🔴 Quiz 3: 진로 상담 RAG 에이전트 완성
# ============================================================
# 📋 목표: 나만의 RAG 에이전트로 진로 질문에 답하기
# ⏱  제한시간: 15분
# TODO: ??? 부분을 채워서 완성하세요!
# ============================================================

# 1️⃣ 나만의 RAG 에이전트 (instructions을 직접 작성!)
my_rag_agent = Agent(
    client=chat_client,
    name="내진로상담AI",
    instructions="""???
    # 힌트: 진로 상담가 역할, 검색 도구 활용 명시, 출처 표시
    """,
    tools=[???]  # search_knowledge_base를 등록하세요
)

print("🎓 나의 진로 상담 RAG 에이전트 시작!")
print("=" * 60)

# 2️⃣ 내가 궁금한 진로 질문 3개!
my_questions = [
    "???",  # 예: "컴퓨터공학과와 인공지능학과의 차이가 뭔가요?"
    "???",  # 예: "수학을 잘 해야 AI 직군으로 취직할 수 있나요?"
    "???",  # 예: "고등학교 때 어떤 활동을 하면 AI 분야 진학에 유리할까요?"
]

for i, q in enumerate(my_questions, 1):
    print(f"\n{'='*60}")
    print(f"❓ 질문 {i}: {q}")
    answer = await my_rag_agent.run(???)
    print(f"🤖 답변:\n{answer}")

In [ ]:
# ============================================================
# 🌟 보너스: RAG + Session 결합 에이전트
# ============================================================

# 1️⃣ 세션 생성
rag_session = await my_rag_agent.???()

print("🌟 RAG + 세션 메모리 결합 에이전트")
print("=" * 60)

# 2️⃣ 대화 1: 자기소개 + 진로 관심사
b1 = "안녕! 나는 고2 소연이야. 장래희망이 AI 연구원이야."
r_b1 = await my_rag_agent.run(b1, ???=rag_session)
print(f"\n👤 소연: {b1}")
print(f"🤖 AI: {r_b1}")

# 3️⃣ 대화 2: 진로 정보 검색 (RAG 작동 확인)
b2 = "AI 연구원이 되려면 어떤 학과에 가야 해?"
r_b2 = await my_rag_agent.run(b2, ???=rag_session)
print(f"\n👤 소연: {b2}")
print(f"🤖 AI: {r_b2}")

# 4️⃣ 대화 3: 메모리 확인 (처음 말한 꿈을 기억하는지)
b3 = "내 꿈이 뭐라고 했지?"
r_b3 = await my_rag_agent.run(b3, ???=rag_session)
print(f"\n👤 소연: {b3}")
print(f"🤖 AI: {r_b3}")

In [ ]:
# ✅ 보너스 정답 예시 (강사 참고용)

# my_rag_agent = Agent(
#     client=chat_client, name="내진로상담AI",
#     instructions="""당신은 고등학생의 진로 상담 전문 AI입니다.
# 질문에 답할 때 search_knowledge_base 도구를 사용해 정확한 정보를 검색하세요.
# 답변 마지막에 반드시 [출처: 참고 자료 N]을 표시하세요.
# 학생의 꿈과 관심사를 기억하고 맞춤형 조언을 드리세요.""",
#     tools=[search_knowledge_base]
# )
# rag_session = await my_rag_agent.create_session()
# r = await my_rag_agent.run("...", session=rag_session)

---
# 🎤 마무리 & 해커톤 예고

## 오늘 배운 것 정리

| 주제 | MAF 1.0.0 핵심 API | 활용 예시 |
|------|-------------------|----------|
| 멀티 에이전트 | `GroupChatBuilder`, `set_select_speakers_func` | 작가+편집자 협업 |
| 세션 메모리 | `create_session()`, `run(..., session=)` | 이름·과목 기억 |
| RAG 에이전트 | `@tool`, `tools=[...]` | 진로 상담 에이전트 |

## 다음 회차: 8회차 해커톤 🔥

```
📅 7월 18일 (토) 10:00~17:00

오전: 디자인 싱킹 워크숍
     - 팀별 문제 정의
     - 아이디어 브레인스토밍

오후: 개발 & 데모 발표
     - 오늘 배운 GroupChat + Session + RAG 모두 활용!
     - 주제: '학교 생활 개선 에이전트'
     - 우수팀 시상 🏆
```

## 해커톤 준비 체크리스트
- [ ] 오늘 노트북 다시 보며 코드 이해하기
- [ ] 팀원과 아이디어 미리 논의하기
- [ ] `data/rag_source.md`에 팀 프로젝트 관련 지식 추가해보기
- [ ] GitHub에 오늘 실습 결과 커밋!

---
*PINE 2기 여러분, 오늘도 정말 수고했습니다! 🌲*